<a href="https://colab.research.google.com/github/angioitoan2409/flood_forecasting/blob/main/ms_checking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
BASE_DIR = "/content/drive/MyDrive/flood_forecasting"
MODEL_DIR = f"{BASE_DIR}/model_inputs"

os.chdir(MODEL_DIR)  # so relative paths like "dtm.tif" work as-is below

# double check the mask files are actually there before loading
print(os.listdir(MODEL_DIR))

['dtm.tif', 'sealed_fraction.tif', 'buildings.tif', 'nonsealed_fraction.tif', 'manning_n.tif', 'radolan_aligned', 'MS5_mask.tif', 'MS2_mask.tif']


In [3]:
import rasterio

with rasterio.open(f"{MODEL_DIR}/dtm.tif") as ref, \
     rasterio.open(f"{MODEL_DIR}/MS2_mask.tif") as ms2:
    print("dtm shape:", ref.shape, "transform:", ref.transform)
    print("MS2 shape:", ms2.shape, "transform:", ms2.transform)
    print("Shapes match:", ref.shape == ms2.shape)
    print("Transforms match:", ref.transform == ms2.transform)

dtm shape: (1600, 1600) transform: | 5.00, 0.00, 414000.00|
| 0.00,-5.00, 5654000.00|
| 0.00, 0.00, 1.00|
MS2 shape: (1600, 1600) transform: | 5.00, 0.00, 414000.00|
| 0.00,-5.00, 5654000.00|
| 0.00, 0.00, 1.00|
Shapes match: True
Transforms match: True


In [4]:
import numpy as np
import rasterio

for name, expected_km2 in [("MS2_mask.tif", 1.3), ("MS5_mask.tif", 0.21)]:
    with rasterio.open(f"{MODEL_DIR}/{name}") as src:
        mask = src.read(1) == 1
        area_km2 = mask.sum() * 25 / 1e6
        print(f"{name}: {mask.sum()} cells = {area_km2:.3f} km² (paper says {expected_km2} km²)")

MS2_mask.tif: 46940 cells = 1.173 km² (paper says 1.3 km²)
MS5_mask.tif: 8338 cells = 0.208 km² (paper says 0.21 km²)
